# Fine-Tune Gemma 3 1B-IT with LoRA & Unsloth

This notebook fine-tunes [google/gemma-3-1b-it](https://huggingface.co/google/gemma-3-1b-it)
for any instruction-following / chat task using **LoRA** and **[Unsloth](https://github.com/unslothai/unsloth)**
for 2x faster training with less memory.

## What you need
- A **CSV file** with `instruction` and `response` columns
- A **GPU runtime** (Colab T4 or better)

## How to customize
1. Upload your CSV and set `data_path` in the config
2. Optionally add a system prompt via `system_prompt`
3. Adjust hyperparameters as needed
4. Run all cells

## 1. Install Dependencies

In [ ]:
!pip install -q --upgrade pip
!pip install -q --no-deps bitsandbytes accelerate xformers peft trl triton cut_cross_entropy unsloth_zoo
!pip install -q sentencepiece protobuf datasets huggingface_hub hf_transfer
!pip install -q unsloth sacrebleu evaluate

## 2. Imports & GPU Setup

In [ ]:
from unsloth import FastModel
from unsloth.chat_templates import get_chat_template

import gc
import json
import os
import random
from dataclasses import asdict, dataclass
from typing import Any, Dict, List, Optional, Tuple

import evaluate
import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm
from transformers import (
    EarlyStoppingCallback,
    Trainer,
    TrainerCallback,
    TrainingArguments,
)

import warnings
warnings.filterwarnings("ignore")

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print(f"PyTorch: {torch.__version__}, CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

if IN_COLAB:
    drive.mount("/content/drive")

## 3. Configuration

**Edit the fields below to match your task.** At minimum, set:
- `data_path` — path to your CSV with `instruction` and `response` columns
- `output_dir` — where to save checkpoints and results
- `system_prompt` — optional system prompt prepended to every example

In [ ]:
@dataclass
class Config:
    # === Data ===
    data_path: str = "/content/drive/MyDrive/data/instructions.csv"  # CSV with 'instruction' and 'response' columns
    output_dir: str = "/content/drive/MyDrive/gemma3-finetune-output"
    test_size: float = 0.1          # fraction held out for val + test
    split_seed: int = 42

    # === Model ===
    model_name: str = "unsloth/gemma-3-1b-it"
    max_seq_length: int = 2048
    load_in_4bit: bool = False  # set True for QLoRA (4-bit quantized base model, less VRAM)
    system_prompt: str = ""  # optional: e.g. "You are a helpful assistant."

    # === Training ===
    per_device_train_batch_size: int = 32
    gradient_accumulation_steps: int = 2
    per_device_eval_batch_size: int = 8
    learning_rate: float = 3e-5
    num_train_epochs: int = 5
    warmup_ratio: float = 0.1
    weight_decay: float = 0.01
    max_grad_norm: float = 1.0
    lr_scheduler_type: str = "cosine"
    logging_steps: int = 100
    eval_steps: int = 100
    save_steps: int = 100
    seed: int = 42

    # === Generation ===
    generation_batch_size: int = 128
    generation_max_new_tokens: int = 256

    # === LoRA ===
    lora_r: int = 32
    lora_alpha: int = 32
    lora_dropout: float = 0.0

    # === Subset (for quick test runs) ===
    subset_ratio: float = 1.0  # set < 1.0 to train on a fraction of data


CONFIG = Config()

print(json.dumps(asdict(CONFIG), indent=2))

## 4. Helpers

In [ ]:
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def build_prompt(instruction: str, system_prompt: str = "") -> str:
    """Build a Gemma 3 chat prompt from an instruction."""
    parts = []
    if system_prompt:
        parts.append(f"<start_of_turn>system\n{system_prompt}<end_of_turn>")
    parts.append(f"<start_of_turn>user\n{instruction}<end_of_turn>")
    parts.append("<start_of_turn>model\n")
    return "\n".join(parts)


def build_training_text(instruction: str, response: str, system_prompt: str = "") -> Tuple[str, str]:
    """Build full training text and the prompt-only prefix."""
    prompt = build_prompt(instruction, system_prompt)
    full_text = f"{prompt}{response}<end_of_turn>\n"
    return full_text, prompt


def clean_generated_text(text: str) -> str:
    """Clean model output by removing chat template artifacts."""
    if text is None:
        return ""
    cleaned = str(text)
    cleaned = cleaned.split("\nmodel\n")[-1].strip()
    cleaned = cleaned.replace("<start_of_turn>model", "")
    cleaned = cleaned.replace("<end_of_turn>", "")
    return cleaned.strip()

## 5. Load & Split Data

Expects a CSV with `instruction` and `response` columns.

In [ ]:
def load_and_split_data(cfg: Config) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    df = pd.read_csv(cfg.data_path)
    assert "instruction" in df.columns and "response" in df.columns, \
        "CSV must have 'instruction' and 'response' columns"

    df = df.dropna(subset=["instruction", "response"])
    df = df[df["instruction"].str.strip() != ""]
    df = df[df["response"].str.strip() != ""]
    df = df.drop_duplicates(subset=["instruction", "response"], keep="first")
    df = df.reset_index(drop=True)
    print(f"Loaded {len(df)} valid examples")

    train_df, temp_df = train_test_split(df, test_size=cfg.test_size * 2, random_state=cfg.split_seed)
    val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=cfg.split_seed)

    train_df = train_df.reset_index(drop=True)
    val_df = val_df.reset_index(drop=True)
    test_df = test_df.reset_index(drop=True)

    print(f"Split: train={len(train_df)}, val={len(val_df)}, test={len(test_df)}")
    return train_df, val_df, test_df


train_df, val_df, test_df = load_and_split_data(CONFIG)

## 6. Tokenize

In [ ]:
def load_tokenizer(cfg: Config) -> Any:
    """Load tokenizer via Unsloth with Gemma 3 chat template."""
    temp_model, tokenizer = FastModel.from_pretrained(
        model_name=cfg.model_name,
        max_seq_length=cfg.max_seq_length,
        load_in_4bit=cfg.load_in_4bit,
        dtype=torch.bfloat16,
    )
    del temp_model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    tokenizer = get_chat_template(tokenizer, chat_template="gemma-3")
    if tokenizer.pad_token is None and tokenizer.eos_token is not None:
        tokenizer.pad_token = tokenizer.eos_token
    return tokenizer


def prepare_dataset(
    split_df: pd.DataFrame,
    tokenizer: Any,
    cfg: Config,
    split_name: str,
) -> Dataset:
    dataset = Dataset.from_pandas(split_df[["instruction", "response"]], preserve_index=False)

    def tokenize(example: Dict[str, Any]) -> Dict[str, Any]:
        full_text, prompt_text = build_training_text(
            instruction=example["instruction"],
            response=example["response"],
            system_prompt=cfg.system_prompt,
        )

        tokenized = tokenizer(
            full_text,
            truncation=True,
            max_length=cfg.max_seq_length,
            padding=False,
            add_special_tokens=False,
        )

        input_ids = tokenized["input_ids"]
        attention_mask = tokenized["attention_mask"]
        labels = list(input_ids)

        # Mask the prompt tokens so the model only learns to generate the response
        prompt_ids = tokenizer(
            prompt_text,
            truncation=True,
            max_length=cfg.max_seq_length,
            padding=False,
            add_special_tokens=False,
        )["input_ids"]

        prefix_len = min(len(prompt_ids), len(labels))
        labels[:prefix_len] = [-100] * prefix_len

        return {"input_ids": input_ids, "attention_mask": attention_mask, "labels": labels}

    prepared = dataset.map(tokenize, remove_columns=dataset.column_names)
    print(f"Tokenized {split_name}: {len(prepared)} examples")
    return prepared


tokenizer = load_tokenizer(CONFIG)
train_dataset = prepare_dataset(train_df, tokenizer, CONFIG, "train")
val_dataset = prepare_dataset(val_df, tokenizer, CONFIG, "val")

## 7. Load Model with LoRA

In [ ]:
def load_model(cfg: Config) -> Tuple[Any, Any]:
    """Load Gemma 3 1B with LoRA via Unsloth."""
    model, tokenizer = FastModel.from_pretrained(
        model_name=cfg.model_name,
        max_seq_length=cfg.max_seq_length,
        load_in_4bit=cfg.load_in_4bit,
        dtype=torch.bfloat16,
    )

    model = FastModel.get_peft_model(
        model,
        finetune_vision_layers=False,
        finetune_language_layers=True,
        finetune_attention_modules=True,
        finetune_mlp_modules=True,
        r=cfg.lora_r,
        lora_alpha=cfg.lora_alpha,
        lora_dropout=cfg.lora_dropout,
        bias="none",
        random_state=cfg.seed,
    )

    tokenizer = get_chat_template(tokenizer, chat_template="gemma-3")
    if tokenizer.pad_token is None and tokenizer.eos_token is not None:
        tokenizer.pad_token = tokenizer.eos_token

    return model, tokenizer


model, tokenizer = load_model(CONFIG)

## 8. Data Collator

In [ ]:
class ChatDataCollator:
    """Pads input_ids, attention_mask, and labels to the longest sequence in the batch."""
    def __init__(self, tokenizer: Any):
        self.tokenizer = tokenizer

    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
        max_len = max(len(f["input_ids"]) for f in features)
        pad_id = self.tokenizer.pad_token_id or self.tokenizer.eos_token_id

        batch: Dict[str, List] = {"input_ids": [], "attention_mask": [], "labels": []}
        for f in features:
            pad_len = max_len - len(f["input_ids"])
            batch["input_ids"].append(list(f["input_ids"]) + [pad_id] * pad_len)
            batch["attention_mask"].append(list(f["attention_mask"]) + [0] * pad_len)
            batch["labels"].append(list(f["labels"]) + [-100] * pad_len)

        return {k: torch.tensor(v) for k, v in batch.items()}

## 9. Training

In [ ]:
class JsonlLogger(TrainerCallback):
    """Logs training metrics to a JSONL file."""
    def __init__(self, path: str):
        self.path = path
        os.makedirs(os.path.dirname(path), exist_ok=True)
        open(self.path, "w").close()

    def on_log(self, args, state, control, logs=None, **kwargs):
        if not logs:
            return
        record = {"step": int(state.global_step)}
        record.update({k: float(v) if isinstance(v, (int, float)) else v for k, v in logs.items()})
        with open(self.path, "a", encoding="utf-8") as f:
            f.write(json.dumps(record) + "\n")

In [ ]:
set_seed(CONFIG.seed)
os.makedirs(CONFIG.output_dir, exist_ok=True)

# Optional: use a subset for quick test runs
if CONFIG.subset_ratio < 1.0:
    n_train = max(1, int(len(train_dataset) * CONFIG.subset_ratio))
    n_val = max(1, int(len(val_dataset) * CONFIG.subset_ratio))
    rng = np.random.default_rng(CONFIG.seed)
    train_dataset = train_dataset.select(sorted(rng.permutation(len(train_dataset))[:n_train].tolist()))
    val_dataset = val_dataset.select(sorted(rng.permutation(len(val_dataset))[:n_val].tolist()))
    print(f"Using subset: train={len(train_dataset)}, val={len(val_dataset)}")

training_args = TrainingArguments(
    output_dir=os.path.join(CONFIG.output_dir, "checkpoints"),
    per_device_train_batch_size=CONFIG.per_device_train_batch_size,
    per_device_eval_batch_size=CONFIG.per_device_eval_batch_size,
    gradient_accumulation_steps=CONFIG.gradient_accumulation_steps,
    warmup_ratio=CONFIG.warmup_ratio,
    num_train_epochs=CONFIG.num_train_epochs,
    learning_rate=CONFIG.learning_rate,
    optim="adamw_torch_fused",
    weight_decay=CONFIG.weight_decay,
    max_grad_norm=CONFIG.max_grad_norm,
    lr_scheduler_type=CONFIG.lr_scheduler_type,
    logging_steps=CONFIG.logging_steps,
    eval_strategy="steps",
    eval_steps=CONFIG.eval_steps,
    save_strategy="steps",
    save_steps=CONFIG.save_steps,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),
    report_to="none",
    seed=CONFIG.seed,
    data_seed=CONFIG.seed,
    remove_unused_columns=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=ChatDataCollator(tokenizer),
    callbacks=[
        EarlyStoppingCallback(early_stopping_patience=2),
        JsonlLogger(os.path.join(CONFIG.output_dir, "train_log.jsonl")),
    ],
)

trainer.train()

## 10. Save the Fine-Tuned Model

In [ ]:
final_model_dir = os.path.join(CONFIG.output_dir, "final_model")
model.save_pretrained(final_model_dir)
tokenizer.save_pretrained(final_model_dir)
print(f"Model saved to: {final_model_dir}")

del trainer, model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

## 11. Evaluate on Test Set

In [ ]:
def load_finetuned_model(cfg: Config, adapter_path: str) -> Tuple[Any, Any]:
    """Load the base model with the fine-tuned LoRA adapter for inference."""
    model, tokenizer = FastModel.from_pretrained(
        model_name=adapter_path,
        max_seq_length=cfg.max_seq_length,
        load_in_4bit=cfg.load_in_4bit,
        dtype=torch.bfloat16,
    )
    tokenizer = get_chat_template(tokenizer, chat_template="gemma-3")
    if tokenizer.pad_token is None and tokenizer.eos_token is not None:
        tokenizer.pad_token = tokenizer.eos_token
    model.eval()
    return model, tokenizer


def generate_responses(
    instructions: List[str],
    model: Any,
    tokenizer: Any,
    cfg: Config,
) -> List[str]:
    """Generate responses for a list of instructions in batches."""
    predictions: List[str] = []

    for i in tqdm(range(0, len(instructions), cfg.generation_batch_size), desc="Generating"):
        batch = instructions[i : i + cfg.generation_batch_size]
        prompts = [build_prompt(inst, cfg.system_prompt) for inst in batch]

        inputs = tokenizer(
            prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=cfg.max_seq_length,
            add_special_tokens=False,
        ).to(model.device)
        input_lens = inputs["attention_mask"].sum(dim=1).tolist()

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=cfg.generation_max_new_tokens,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
            )

        for out, in_len in zip(outputs, input_lens):
            decoded = tokenizer.decode(out[int(in_len):], skip_special_tokens=True).strip()
            predictions.append(clean_generated_text(decoded))

    return predictions

In [ ]:
eval_model, eval_tokenizer = load_finetuned_model(CONFIG, final_model_dir)

instructions = test_df["instruction"].astype(str).tolist()
references = test_df["response"].astype(str).tolist()
predictions = generate_responses(instructions, eval_model, eval_tokenizer, CONFIG)

# Save predictions
results_df = test_df.copy()
results_df["prediction"] = predictions
results_df.to_csv(os.path.join(CONFIG.output_dir, "test_predictions.csv"), index=False)

print(f"\nGenerated {len(predictions)} predictions")
print(f"Results saved to: {os.path.join(CONFIG.output_dir, 'test_predictions.csv')}")

# Show a few examples
for i in range(min(3, len(predictions))):
    print(f"\n{'='*60}")
    print(f"Instruction: {instructions[i][:200]}")
    print(f"Expected:    {references[i][:200]}")
    print(f"Predicted:   {predictions[i][:200]}")

del eval_model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

## 12. Inference Example

Use the fine-tuned model to generate responses for new instructions:

In [ ]:
model, tokenizer = load_finetuned_model(CONFIG, final_model_dir)

sample_instructions = [
    "Explain quantum computing in simple terms.",
    "Write a short poem about machine learning.",
]

responses = generate_responses(sample_instructions, model, tokenizer, CONFIG)

for inst, resp in zip(sample_instructions, responses):
    print(f"  Instruction: {inst}")
    print(f"  Response:    {resp}")
    print()